In [0]:
import pandas as pd
import time
from pyspark.sql import functions as f
from pyspark.sql import types as t
from src.utils.validations import *
from src.utils.file import *
from datetime import datetime
from pyspark.sql import SparkSession
import os
from zoneinfo import ZoneInfo
from quality.engine.rules import Rules
from datetime import datetime
from zoneinfo import ZoneInfo
from functools import reduce
from quality.pipelines.quality_discagem import Quality as qa

In [0]:
def read_csv(path: str, use_spark=False, **kwargs):

        try:

            if use_spark:

                df = (
                    spark.read
                    .options(**kwargs)
                    .csv(path)
                    .withColumn(
                        "source_file",
                        f.element_at(
                            f.split(
                                f.col("_metadata.file_path"),
                                "/"
                            ),
                            -1
                        )
                    )
                )

                # força validação
                df.limit(1).collect()

            else:

                df = pd.read_csv(path, **kwargs)

                df["source_file"] = os.path.basename(path)

            return df

        except Exception as e:

            print(f"Falha ao ler o arquivo {path}: {e}")

            return None


In [0]:

dir_path = r"/Volumes/workspace/callcenter/disc_volumes/landing/"
df_processed_file = spark.table(r"workspace.quality.execucao_arquivo")

--- VALIDA EXISTENCIA DE ARQUIVO DO DIA ATUAL

In [0]:


data_hoje = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).strftime("%Y%m%d")


files = dbutils.fs.ls(dir_path)
dfs = []

for file in files:

    if f"discagem_{data_hoje}_" in file.name:

        df = read_csv(path=file.path, use_spark=True, sep=";", header=True)
        dfs.append(df)


df_concat = reduce(
    lambda df1, df2: df1.unionByName(df2)
    , dfs
)


In [0]:
%sql
select * from workspace.quality.erros_arquivo

In [0]:
%sql
select * from workspace.quality.erros_linha

In [0]:
%sql
select * from workspace.quality.execucao_arquivo

In [0]:
tabelas_qa = ["workspace.quality.erros_arquivo", "workspace.quality.erros_linha", "workspace.quality.execucao_arquivo"]

for tabela in tabelas_qa:
    spark.sql("DROP TABLE IF EXISTS {tabela}")
